# Alpha 模板展开与批量回测

基于 `Course1/Course1_模板合集/可以尝试使用的Alpha模板.md` 自动提取模板、展开占位符并批量提交 WorldQuant BRAIN simulation。

In [6]:
from __future__ import annotations

import itertools
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from requests.auth import HTTPBasicAuth

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

In [15]:
def find_workspace_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "Course1").exists() and (p / "Course2").exists():
            return p
    raise RuntimeError("未找到 workspace 根目录（需同时包含 Course1 与 Course2）")

ROOT = find_workspace_root(Path.cwd())
TEMPLATE_MD = ROOT / "Course1" / "Course1_模板合集" / "可以尝试使用的Alpha模板.md"
OUTPUT_DIR = ROOT / "Course2" / "Course2_code" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BRAIN_USERNAME = os.getenv("BRAIN_USERNAME", "3052724412@qq.com")
BRAIN_PASSWORD = os.getenv("BRAIN_PASSWORD", "SuerJA8HCSvMV")

DATASET_ID = "fundamental6"
DATA_TYPES = ["MATRIX", "VECTOR", "GROUP"]

SETTINGS = {
    "type": "REGULAR",
    "settings": {
        "instrumentType": "EQUITY",
        "region": "USA",
        "universe": "TOP3000",
        "delay": 1,
        "decay": 10,
        "neutralization": "INDUSTRY",
        "truncation": 0.08,
        "pasteurization": "ON",
        "unitHandling": "VERIFY",
        "nanHandling": "ON",
        "language": "FASTEXPR",
        "visualization": False
    }
}

MAX_EXPANDED_PER_TEMPLATE = 4
MAX_TOTAL_TO_SIM = 40
REQUEST_TIMEOUT = 30

ROOT, TEMPLATE_MD, OUTPUT_DIR, DATASET_ID, DATA_TYPES

(WindowsPath('d:/DBC/量化实战课_20260412'),
 WindowsPath('d:/DBC/量化实战课_20260412/Course1/Course1_模板合集/可以尝试使用的Alpha模板.md'),
 WindowsPath('d:/DBC/量化实战课_20260412/Course2/Course2_code/outputs'),
 'fundamental6',
 ['MATRIX', 'VECTOR', 'GROUP'])

In [16]:
def build_session(username: str, password: str) -> requests.Session:
    sess = requests.Session()
    sess.auth = HTTPBasicAuth(username, password)
    return sess

def authenticate(sess: requests.Session) -> dict:
    r = sess.post("https://api.worldquantbrain.com/authentication", timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()

def get_datafields(
    sess: requests.Session,
    dataset_id: str,
    data_type: str,
    max_retries_per_page: int = 6
) -> list[str]:
    out = []
    offset = 0
    while True:
        url = (
            "https://api.worldquantbrain.com/data-fields?"
            f"instrumentType={SETTINGS['settings']['instrumentType']}&region={SETTINGS['settings']['region']}"
            f"&delay={SETTINGS['settings']['delay']}&universe={SETTINGS['settings']['universe']}"
            f"&dataset.id={dataset_id}&limit=50&offset={offset}&type={data_type}"
        )

        retries = 0
        while True:
            r = sess.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code != 429:
                break
            retries += 1
            if retries > max_retries_per_page:
                r.raise_for_status()
            wait_s = float(r.headers.get("Retry-After", 2))
            time.sleep(max(wait_s, 1.0))

        if r.status_code >= 400:
            raise requests.HTTPError(f"HTTP {r.status_code}: {r.text[:300]}", response=r)

        payload = r.json()
        rows = payload.get("results", [])
        out.extend([x.get("id") for x in rows if x.get("id")])
        if len(rows) < 50:
            break
        offset += 50
        time.sleep(0.5)
    return out

def get_fundamental6_datafields(sess: requests.Session, data_types: list[str]) -> tuple[list[str], dict[str, int], dict[str, str]]:
    merged = []
    type_counts = {}
    type_errors = {}

    for dt in data_types:
        try:
            vals = get_datafields(sess, dataset_id=DATASET_ID, data_type=dt)
            type_counts[dt] = len(vals)
            merged.extend(vals)
        except Exception as ex:
            type_counts[dt] = 0
            type_errors[dt] = str(ex)

    dedup = sorted(set(merged))
    return dedup, type_counts, type_errors

sess = build_session(BRAIN_USERNAME, BRAIN_PASSWORD)
auth_info = authenticate(sess)
fields, type_counts, type_errors = get_fundamental6_datafields(sess, DATA_TYPES)

print("auth ok", auth_info)
print("dataset:", DATASET_ID)
print("data type counts:", type_counts)
if type_errors:
    print("data type errors:", type_errors)
print("fetched unique fields:", len(fields))

auth ok {'user': {'id': 'FL31207'}, 'token': {'expiry': 14400.0}, 'permissions': ['TUTORIAL']}
dataset: fundamental6
data type counts: {'MATRIX': 574, 'VECTOR': 312, 'GROUP': 0}
fetched unique fields: 886


In [17]:
def clean_template_line(line: str) -> str:
    s = line.strip()
    s = s.replace("，", ",").replace("：", ":").replace("；", ";")
    s = re.sub(r"\s+", " ", s)
    return s

def extract_templates(md_text: str) -> list[str]:
    text = re.sub(r"\\\\\s*\n", "", md_text)
    lines = [clean_template_line(x) for x in text.splitlines()]

    keywords = [
        "group_neutralize", "regression_neut", "vector_neut", "trade_when",
        "ts_", "group_", "rank(", "power(", "if_else("
    ]

    templates = []
    for ln in lines:
        if not ln:
            continue
        if "关注" in ln or "已编辑" in ln or "shout out" in ln.lower():
            continue
        if "=" in ln and ln.count("(") == 0:
            continue
        if any(k in ln for k in keywords) and "(" in ln and ")" in ln:
            templates.append(ln.rstrip(";"))

    dedup = []
    seen = set()
    for t in templates:
        k = re.sub(r"\s+", "", t)
        if k not in seen:
            seen.add(k)
            dedup.append(t)
    return dedup

def placeholder_candidates(fields: list[str]) -> dict[str, list[str]]:
    analyst = [f for f in fields if "anl" in f.lower() or "analyst" in f.lower()]
    fundamental = [f for f in fields if "fnd" in f.lower() or "mdl" in f.lower()]

    fallback = fields[:8] if fields else ["close", "volume"]

    return {
        "data": fallback[:4],
        "datafield": fallback[:4],
        "Fundamental": (fundamental[:4] or fallback[:4]),
        "ANALYST": (analyst[:4] or fallback[:4]),
        "RETURNS": ["returns"],
        "returns": ["returns"],
        "i": (fallback[:2] or ["close"]),
        "j": (fallback[2:4] or fallback[:2] or ["volume"]),
        "sell_order": (fallback[:2] or ["volume"]),
        "buy_order": (fallback[2:4] or fallback[:2] or ["sharesout"])
    }

def expand_template(tpl: str, cand_map: dict[str, list[str]], limit_per_tpl: int = 4) -> list[str]:
    token_re = re.compile(r"\{([^{}]+)\}")
    tokens = token_re.findall(tpl)

    dynamic_map = {
        "ts_opr_1": ["ts_mean", "ts_rank"],
        "ts_opr_2": ["ts_corr", "ts_covariance"],
        "group_opr": ["group_neutralize", "group_rank"],
        "vector_opr": ["vec_avg", "vec_max"],
        "pv_field": ["close", "vwap"],
        "vol_field": ["volume", "sharesout"],
        "days": ["63", "126"],
        "days1": ["20", "63"],
        "days2": ["5", "10"],
        "grouping": ["industry", "sector"],
        "Arithmetich_or Transformational_op": ["rank", "signed_power"],
        "ts_compare_op": ["ts_corr", "ts_covariance"],
        "Company Fundamental Data for Equity": ["mdl175_bp", "fnd72_s_pit_or_cf_q_cf_net_inc"],
        "Price Volume Data for Equity": ["close", "volume"],
    }

    if not tokens:
        return [tpl]

    choices = []
    for tk in tokens:
        if tk in dynamic_map:
            choices.append(dynamic_map[tk])
        elif tk in cand_map:
            choices.append(cand_map[tk])
        else:
            choices.append(["close"])

    expanded = []
    for combo in itertools.product(*choices):
        s = tpl
        for tk, val in zip(tokens, combo):
            s = s.replace("{" + tk + "}", str(val))
        expanded.append(s)
        if len(expanded) >= limit_per_tpl:
            break

    return expanded

raw_md = TEMPLATE_MD.read_text(encoding="utf-8")
base_templates = extract_templates(raw_md)
cand_map = placeholder_candidates(fields)

expanded = []
for idx, tpl in enumerate(base_templates, start=1):
    exps = expand_template(tpl, cand_map, limit_per_tpl=MAX_EXPANDED_PER_TEMPLATE)
    for e in exps:
        expanded.append({"template_id": idx, "template": tpl, "expression": e})

expanded_df = pd.DataFrame(expanded).drop_duplicates(subset=["expression"]).reset_index(drop=True)
if len(expanded_df) > MAX_TOTAL_TO_SIM:
    expanded_df = expanded_df.head(MAX_TOTAL_TO_SIM).copy()

expanded_path = OUTPUT_DIR / "expanded_templates.csv"
expanded_df.to_csv(expanded_path, index=False, encoding="utf-8-sig")

print("base templates:", len(base_templates))
print("expanded for simulation:", len(expanded_df))
expanded_df.head(10)

base templates: 187
expanded for simulation: 40


,template_id,template,expression
0,1,"vec_avg({data}),sector),bucket(rank(cap),range...","vec_avg(assets),sector),bucket(rank(cap),range..."
1,1,"vec_avg({data}),sector),bucket(rank(cap),range...","vec_avg(assets_curr),sector),bucket(rank(cap),..."
2,1,"vec_avg({data}),sector),bucket(rank(cap),range...","vec_avg(bookvalue_ps),sector),bucket(rank(cap)..."
3,1,"vec_avg({data}),sector),bucket(rank(cap),range...","vec_avg(capex),sector),bucket(rank(cap),range=..."
4,2,"group_neutralize(group_zscore(cap,sector),buck...","group_neutralize(group_zscore(cap,sector),buck..."
5,3,"ts_ir(returns-group_median(returns,sector),126))","ts_ir(returns-group_median(returns,sector),126))"
6,4,fear = ts_mean(abs(returns - group_mean(return...,fear = ts_mean(abs(returns - group_mean(return...
7,5,-group_neutralize(fear*group_normalize(ts_deca...,-group_neutralize(fear*group_normalize(ts_deca...
8,6,"-ts_percentage(vec_count({data}),60,percentage...","-ts_percentage(vec_count(assets),60,percentage..."
9,6,"-ts_percentage(vec_count({data}),60,percentage...","-ts_percentage(vec_count(assets_curr),60,perce..."


In [18]:
def submit_simulation(sess: requests.Session, expression: str) -> dict:
    payload = {"type": SETTINGS["type"], "settings": SETTINGS["settings"], "regular": expression}

    try:
        r = sess.post("https://api.worldquantbrain.com/simulations", json=payload, timeout=REQUEST_TIMEOUT)
        if r.status_code >= 400:
            return {"status": "submit_error", "error": f"HTTP {r.status_code}: {r.text[:300]}"}
        location = r.headers.get("Location")
        if not location:
            return {"status": "submit_no_location", "error": "Missing Location header"}

        while True:
            pr = sess.get(location, timeout=REQUEST_TIMEOUT)
            if pr.status_code >= 400:
                return {"status": "poll_error", "location": location, "error": f"HTTP {pr.status_code}: {pr.text[:300]}"}

            retry_after = float(pr.headers.get("Retry-After", 0))
            if retry_after == 0:
                data = pr.json()
                return {
                    "status": "done",
                    "location": location,
                    "alpha_id": data.get("alpha"),
                    "result": data
                }

            time.sleep(max(retry_after, 1.0))

    except Exception as ex:
        return {"status": "exception", "error": str(ex)}

results = []
for i, row in expanded_df.iterrows():
    expr = row["expression"]
    resp = submit_simulation(sess, expr)
    results.append({
        "idx": i + 1,
        "template_id": row["template_id"],
        "expression": expr,
        "status": resp.get("status"),
        "alpha_id": resp.get("alpha_id"),
        "location": resp.get("location"),
        "error": resp.get("error"),
        "result": json.dumps(resp.get("result", {}), ensure_ascii=False)
    })

    if (i + 1) % 5 == 0:
        print(f"progress: {i + 1}/{len(expanded_df)}")

results_df = pd.DataFrame(results)
results_path = OUTPUT_DIR / "simulation_status.csv"
results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

json_path = OUTPUT_DIR / "simulation_status.json"
json_path.write_text(results_df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")

print("saved:", results_path)
results_df.head(10)

progress: 5/40
progress: 10/40
progress: 15/40
progress: 20/40
progress: 25/40
progress: 30/40
progress: 35/40
progress: 40/40
saved: d:\DBC\量化实战课_20260412\Course2\Course2_code\outputs\simulation_status.csv


,idx,template_id,expression,status,alpha_id,location,error,result
0,1,1,"vec_avg(assets),sector),bucket(rank(cap),range...",done,None,https://api.worldquantbrain.com/simulations/2g...,None,"{""id"": ""2gtcVDcB154T99115hgCGke9"", ""type"": ""RE..."
1,2,1,"vec_avg(assets_curr),sector),bucket(rank(cap),...",done,None,https://api.worldquantbrain.com/simulations/eq...,None,"{""id"": ""eq5xjdVj5e0bGJ10sPEWWjs"", ""type"": ""REG..."
2,3,1,"vec_avg(bookvalue_ps),sector),bucket(rank(cap)...",done,None,https://api.worldquantbrain.com/simulations/46...,None,"{""id"": ""46dFdb6M95jSakM10nPBj11W"", ""type"": ""RE..."
3,4,1,"vec_avg(capex),sector),bucket(rank(cap),range=...",done,None,https://api.worldquantbrain.com/simulations/4n...,None,"{""id"": ""4nu00563T4Jmc47dG0QYMgp"", ""type"": ""REG..."
4,5,2,"group_neutralize(group_zscore(cap,sector),buck...",done,None,https://api.worldquantbrain.com/simulations/44...,None,"{""id"": ""446ZDc6EN4y0brDc7bNCDZy"", ""type"": ""REG..."
5,6,3,"ts_ir(returns-group_median(returns,sector),126))",done,None,https://api.worldquantbrain.com/simulations/29...,None,"{""id"": ""29CWfTfn64ps9vxZHfJlcXY"", ""type"": ""REG..."
6,7,4,fear = ts_mean(abs(returns - group_mean(return...,done,None,https://api.worldquantbrain.com/simulations/3Q...,None,"{""id"": ""3QHxOQ4EI4DA9jmTuqkbCTz"", ""type"": ""REG..."
7,8,5,-group_neutralize(fear*group_normalize(ts_deca...,done,None,https://api.worldquantbrain.com/simulations/4h...,None,"{""id"": ""4h5xKB7pq4FmbP1PUprzuuJ"", ""type"": ""REG..."
8,9,6,"-ts_percentage(vec_count(assets),60,percentage...",done,None,https://api.worldquantbrain.com/simulations/1T...,None,"{""id"": ""1TchCnaL85799znvZqVrHCc"", ""type"": ""REG..."
9,10,6,"-ts_percentage(vec_count(assets_curr),60,perce...",done,None,https://api.worldquantbrain.com/simulations/ho...,None,"{""id"": ""hoCJ938h4DjcD81bk9UJUmj"", ""type"": ""REG..."


In [19]:
results_df = results_df.copy()
results_df["is_done"] = results_df["status"].eq("done")
results_df["has_result"] = results_df["result"].fillna("{}").ne("{}")
results_df["has_location"] = results_df["location"].notna()

summary = (
    results_df.groupby("status", dropna=False)
    .agg(
        cnt=("status", "count"),
        done_cnt=("is_done", "sum"),
        with_result_cnt=("has_result", "sum"),
        with_location_cnt=("has_location", "sum"),
    )
    .reset_index()
    .sort_values("cnt", ascending=False)
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
summary_path = OUTPUT_DIR / f"simulation_summary_{stamp}.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("summary file:", summary_path)
summary

summary file: d:\DBC\量化实战课_20260412\Course2\Course2_code\outputs\simulation_summary_20260419T141711Z.csv


,status,cnt,done_cnt,with_result_cnt,with_location_cnt
0,done,40,40,40,40
